# Example 5: Multi-page plan

The drawing from papermodels' Example 5: a rotated, multi-page plan in which a column above
(`CPL0.4`) transfers to `FB0.2` rather than to every element it touches.

**Status:** papermodels 0.21 cannot build this graph yet. `GeometryGraph.from_pdf_file` raises
`KeyError: 'element'` in `GeometryGraph.collector_elements` (`geometry_graph.py:106`), because a
node in the graph has no element. The papermodels notebook for this example fails in the same
way. The cells below show the error. The rest of the notebook is the loadtracer process for
when the graph builds.

The PDF (37 MB) is not copied into this repository. Set `PAPERMODELS_EXAMPLES` to the
papermodels `Examples` directory, or keep papermodels next to loadtracer.

In [1]:
%matplotlib inline
import sys
import warnings
from decimal import Decimal

import pandas as pd
from papermodels.datatypes.geometry_graph import GeometryGraph
from papermodels.datatypes.joist_models import JoistArrayModel
from loadtracer.load_tracer import LoadTracer

sys.path.insert(0, "..")
from helpers import (
    bottom_members,
    compare_modes,
    load_check,
    load_table,
    occupancies_in,
    quietly,
    reaction_table,
)

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_rows = 40

## 1. Build the graph (papermodels)

In [2]:
import os
import pathlib

examples = pathlib.Path(os.environ.get("PAPERMODELS_EXAMPLES", "../../../papermodels/Examples"))
pdf_path = examples / "Example 5" / "multi-page-plan.pdf"
if not pdf_path.exists():
    raise FileNotFoundError(
        f"{pdf_path} not found. Set PAPERMODELS_EXAMPLES to the papermodels Examples directory."
    )

In [3]:
graph = GeometryGraph.from_pdf_file(pdf_path, scale=Decimal(1 / 72 * 4))
graph.assign_collector_behaviour(JoistArrayModel, spacing=1.0)

KeyError: 'element'

In [ ]:
graph.plot_annotations(1)

## 2. Occupancies

loadtracer needs the loads of each occupancy in the drawing, as pressures by load case.
These are the occupancies that the loading areas use:

In [ ]:
occupancies_in(graph)

The drawing is in feet, so these pressures (psf) give loads in lb/ft and lb.

In [ ]:
occupancies = {
    "residential": {"D": 15.0, "L": 40.0},
    "roof": {"D": 20.0, "S": 35.0},
}

## 3. Run loadtracer

Every element is analysed from the top of the graph down. Only the graph and the occupancies
are needed; element heights and floor elevations are not inputs. `artifacts/` receives the
analysis models (`models.json`) and the reactions (`reaction_forces.json`).

In [ ]:
lt = LoadTracer(geometry_graph=graph, occupancies=occupancies)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    quietly(lt.execute_analyses, artifacts_dir="artifacts")

print(f"{len(lt.reactions)} members analysed, {len(lt.errors)} errors")
for w in caught:
    print(f"{type(w.message).__name__}: {w.message}")

All of the load applied to the building reaches the bottom of the graph:

In [ ]:
load_check(lt)

`lt.issues` lists input that loadtracer corrected so that the rundown could continue
(for example, load drawn past the end of a member, which is moved to the member end):

In [ ]:
{tag: issues for tag, issues in lt.issues.items()}

## 4. One member, end to end: `FB0.2`

`FB0.2` receives the point load from the column `CPL0.4` above.

In [ ]:
pd.DataFrame(lt.converted_joist_loads("FB0.2"))

The loads on `FB0.2` after expansion, by load case:

In [ ]:
load_table(lt, "FB0.2")

The loads in the Pynite model of `FB0.2`:

In [ ]:
member = lt.models["FB0.2"]["members"]["FB0.2"]
pd.DataFrame(member["DistLoads"] + member["PtLoads"])

The reactions of `FB0.2` (positive downwards on the supports):

In [ ]:
reaction_table(lt, ["FB0.2"])

## 5. Joist loads as point loads or distributed loads

With `joist_loads="point"` each joist reaction is applied as a point load, which is the most
detailed representation. The total load is the same either way. Individual reactions are the
same for uniform joist reactions and differ more the more the joist reactions vary; arrays that
vary by 2:1 or more are listed in the warnings in section 3. For those, check the point load
results.

In [ ]:
lt_point = LoadTracer(geometry_graph=graph, occupancies=occupancies, joist_loads="point")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    quietly(lt_point.execute_analyses)

pd.concat({"point": load_check(lt_point)["at the bottom"], "distributed": load_check(lt)["at the bottom"]}, axis=1)

In [ ]:
compare_modes(lt_point, lt, ["FB0.2"])

## 6. Reactions at the bottom

The loads delivered to the elements at the bottom of the graph (the foundation):

In [ ]:
reaction_table(lt, bottom_members(lt))